# Language Distribution
Summarize detected language and block distributions.

## Data Load
Load all semantic blocks from the parsed chat files.

In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm

FUNCTIONAL_TYPES = {
    "text",
    "details",
    "diff",
    "think",
    "read-file",
    "tool-use",
    "unknown",
    "details",
}

block_records = []

for sha in tqdm(sorted(os.listdir("../data/parsed_chats"))):
    # Skip files that are not JSON
    if not (sha.endswith(".json") or sha.endswith(".md.json")):
        continue

    file_path = f"../data/parsed_chats/{sha}"
    with open(file_path, encoding="utf-8") as f:
        chat = json.load(f)

    chat_id = (
        sha[: -len(".md.json")] if sha.endswith(".md.json") else sha[: -len(".json")]
    )

    for i, msg in enumerate(chat["messages"]):
        aggregated_blocks = []
        for block_group in msg["blocks"]:
            aggregated_blocks.extend(block_group)

        for j, block in enumerate(aggregated_blocks):
            block_type = block["type"].split(":", 1)[0]
            block_group = "func" if block_type in FUNCTIONAL_TYPES else "lang"
            block_records.append(
                {
                    "sha": chat_id,
                    "msg_role": msg["role"],
                    "msg_index": i,
                    "block_index": j,
                    "block_type": block_type,
                    "block_group": block_group,
                    "content": block["content"],
                }
            )

df_blocks = pd.DataFrame(block_records)

100%|██████████| 11655/11655 [00:15<00:00, 752.48it/s]


Load precomputed text and diff language results.

In [2]:
print("Loading previously detected text languages...")
df_text = pd.read_json("../data/detected_langs/all_texts_with_lang.jsonl", orient="records", lines=True)
print("Loading previously detected programming languages...")
df_diff = pd.read_json("../data/detected_langs/all_diffs_with_pl.jsonl", orient="records", lines=True)

Loading previously detected text languages...
Loading previously detected programming languages...


## Block Type Distribution
Show the distribution of block types within `func` blocks.

In [3]:
df_blocks[df_blocks["block_group"] == "func"].value_counts("block_type").sort_values(ascending=False)

block_type
text         568476
details      173631
diff         126376
think         50654
read-file     46666
tool-use      42905
unknown       41993
Name: count, dtype: int64

Show the distribution of programming languages detected in `lang` blocks.

In [4]:
lang_counts = df_blocks.loc[
    df_blocks["block_group"].eq("lang"),
    "block_type",
].value_counts()

merged_counts = (
    lang_counts.groupby(lang_counts.index.where(lang_counts >= 100, "other"))
    .sum()
    .sort_values(ascending=False)
)

pct = merged_counts / merged_counts.sum() * 100
max_count = int(merged_counts.max()) if not merged_counts.empty else 0
bar_width = 40

print("block_type |  count |    pct | scale")
print("-" * 78)
for block_type, count in merged_counts.items():
    ticks = "#" * int(round((count / max_count) * bar_width)) if max_count else ""
    print(f"{block_type:<10} | {count:>6} | {pct[block_type]:6.2f}% | {ticks}")

block_type |  count |    pct | scale
------------------------------------------------------------------------------
bash       |  55700 |  45.84% | ########################################
json       |  26077 |  21.46% | ###################
python     |   6643 |   5.47% | #####
typescript |   6350 |   5.23% | #####
javascript |   4347 |   3.58% | ###
other      |   3162 |   2.60% | ##
java       |   2273 |   1.87% | ##
powershell |   1535 |   1.26% | #
markdown   |   1370 |   1.13% | #
tsx        |   1288 |   1.06% | #
css        |   1238 |   1.02% | #
yaml       |   1057 |   0.87% | #
js         |   1019 |   0.84% | #
ts         |    891 |   0.73% | #
html       |    877 |   0.72% | #
dart       |    841 |   0.69% | #
sql        |    605 |   0.50% | 
mermaid    |    574 |   0.47% | 
cpp        |    500 |   0.41% | 
swift      |    497 |   0.41% | 
rust       |    464 |   0.38% | 
sh         |    450 |   0.37% | 
vue        |    426 |   0.35% | 
c          |    377 |   0.31% | 
zsh    

## User Language Distribution
Show top detected natural languages in user messages.

In [5]:
with open("lang_code_map.json", encoding="utf-8") as f:
    lang_code_map = json.load(f)

top_n = 20
lang_counts = df_text["lang"].value_counts().sort_values(ascending=False)
top_counts = lang_counts.head(top_n)
total = int(lang_counts.sum())
max_count = int(top_counts.max()) if not top_counts.empty else 0
bar_width = 40

print(f"Top {top_n} detected languages (total={total})")
print("code | language        |  count |    pct | scale")
print("-" * 92)

for code, count in top_counts.items():
    info = lang_code_map.get(code, {})
    name = info.get("name", "Unknown")
    pct = (count / total * 100) if total else 0.0
    ticks = "#" * int(round((count / max_count) * bar_width)) if max_count else ""
    print(f"{code:>4} | {name:<15} | {count:>6} | {pct:6.2f}% | {ticks}")

Top 20 detected languages (total=79690)
code | language        |  count |    pct | scale
--------------------------------------------------------------------------------------------
  en | English         |  47581 |  59.71% | ########################################
  zh | Chinese         |  14731 |  18.49% | ############
  ja | Japanese        |   6608 |   8.29% | ######
  es | Spanish         |   2843 |   3.57% | ##
  de | German          |   1433 |   1.80% | #
  fr | French          |    728 |   0.91% | #
  fa | Persian         |    557 |   0.70% | 
  vi | Vietnamese      |    547 |   0.69% | 
  pt | Portuguese      |    546 |   0.69% | 
  la | Latin           |    546 |   0.69% | 
  ru | Russian         |    410 |   0.51% | 
  th | Thai            |    271 |   0.34% | 
  it | Italian         |    261 |   0.33% | 
  pl | Polish          |    229 |   0.29% | 
  ko | Korean          |    203 |   0.25% | 
  ca | Catalan         |    188 |   0.24% | 
  uk | Ukrainian       |    185 |   

## Diff PL Distribution
Show top detected programming languages in diff blocks.

In [6]:
top_n = 20
pl_counts = df_diff["pl"].value_counts().sort_values(ascending=False)
top_pl_counts = pl_counts.head(top_n)
total = int(pl_counts.sum())
max_count = int(top_pl_counts.max()) if not top_pl_counts.empty else 0
bar_width = 40

print(f"Top {top_n} detected PLs in diff blocks (total={total})")
print("                pl |  count |    pct | scale")
print("-" * 86)

for pl, count in top_pl_counts.items():
    pct = (count / total * 100) if total else 0.0
    ticks = "#" * int(round((count / max_count) * bar_width)) if max_count else ""
    print(f"{pl:>18} | {count:>6} | {pct:6.2f}% | {ticks}")

Top 20 detected PLs in diff blocks (total=109763)
                pl |  count |    pct | scale
--------------------------------------------------------------------------------------
          gdscript |  29208 |  26.61% | ########################################
            python |  15125 |  13.78% | #####################
             scdoc |  11051 |  10.07% | ###############
      transact-sql |   9505 |   8.66% | #############
               xml |   8028 |   7.31% | ###########
         text only |   6326 |   5.76% | #########
         css+lasso |   5880 |   5.36% | ########
            carbon |   5255 |   4.79% | #######
            genshi |   2972 |   2.71% | ####
    actionscript 3 |   2649 |   2.41% | ####
   tera term macro |   2476 |   2.26% | ###
       objective-c |   1835 |   1.67% | ###
             mysql |   1628 |   1.48% | ##
            scilab |   1153 |   1.05% | ##
           verilog |   1031 |   0.94% | #
  xml+django/jinja |    843 |   0.77% | #
               ecl